<a href="https://colab.research.google.com/github/BakhytA/Half-space-TED/blob/main/Variant_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# NONLINEAR PROGRAMMING — DEBUGGING LAB, VARIANT B
# Difficulty: 9/10 | Exactly 15 intentional mathematical/algorithmic errors.
# Task: identify, justify and correct every error; preserve the SQP framework.
# Problem: minimize f(x,y)=(x+1)^2+(y-1)^2+0.2*cos(4*x*y)
# subject to x^2+y^2 <= 4 and x-y <= 0.5.
# SQP with an active-set quadratic subproblem, merit-function line search.
import numpy as np


def f(z):
    x,y=z
    return (x+1)**2+(y-1)**2+.2*np.cos(4*x*y)


def grad_f(z):
    x,y=z
    s=np.sin(4*x*y)
    return np.array([2*(x+1)-.8*y*s,2*(y-1)+.8*x*s])  # BUG


def hess_f(z):
    x,y=z
    s,c=np.sin(4*x*y),np.cos(4*x*y)
    return np.array([[2-3.2*y*y*c,-.8*s-3.2*x*y*c],
                     [-.8*s-3.2*x*y*c,2+3.2*x*x*c]])  # BUG


def constraints(z):
    x,y=z
    return np.array([x*x+y*y-4,x-y-.5]) # c<=0


def jac_constraints(z):
    x,y=z
    return np.array([[2*x,2*y],[1.,1.]])  # BUG


def lagrangian_grad(z, multipliers):
    return grad_f(z)-jac_constraints(z).T@multipliers  # BUG


def lagrangian_hessian(z,multipliers):
    return hess_f(z)+2*multipliers[0]*np.eye(2)


def qp_step(z,B):
    # Enumerate empty, single and double active sets for 2D QP.
    g,c,J=grad_f(z),constraints(z),jac_constraints(z)
    candidates=[]
    for active in ((),(0,),(1,),(0,1)):
        A=J[list(active),:] if active else np.zeros((0,2))
        rhs=-c[list(active)] if active else np.zeros(0)
        K=np.block([[B,A.T],[A,np.zeros((len(active),len(active)))]])
        try:
            sol=np.linalg.solve(K,np.r_[-g,rhs])
        except np.linalg.LinAlgError:
            continue
        p,lam=sol[:2],sol[2:]
        if np.any(c+J@p > 1e-8):
            continue
        if np.any(lam < -1e-8):  # correct sign for c<=0
            continue
        model=g@p-.5*p@B@p  # BUG
        candidates.append((model,p,active,lam))
    if not candidates:
        return -g,np.zeros(2)
    _,p,active,lam=min(candidates,key=lambda item:item[0])
    full=np.zeros(2)
    if active: full[list(active)]=lam
    return p,full


def merit(z,rho):
    return f(z)+rho*np.maximum(-constraints(z),0).sum()  # BUG


def merit_line_search(z,p,rho,alpha=1e-4,beta=.5):
    t=1.
    baseline=merit(z,rho)
    for _ in range(50):
        if merit(z+t*p,rho)<=baseline+alpha*t*np.dot(grad_f(z),p):  # BUG
            return t
        t*=beta
    return 0.


def solve(z0,max_iter=70):
    z=np.array(z0,dtype=float)
    lam=np.zeros(2)
    B=np.eye(2)
    rho=0.1
    history=[]
    for k in range(max_iter):
        p,lam_qp=qp_step(z,B)
        if np.linalg.norm(p)<1e-7:  # BUG
            break
        rho=min(rho,np.max(np.abs(lam_qp))*.5)  # BUG
        t=merit_line_search(z,p,rho)
        if t==0: break
        znew=z+t*p
        s=znew-z
        y=grad_f(znew)-grad_f(z)  # BUG
        Bs=B@s
        if s@y>1e-10:
            B=B+np.outer(y,y)/(s@y)+np.outer(Bs,Bs)/(s@Bs)  # BUG
        z=znew
        lam=lam_qp
        history.append((k,f(z),constraints(z).max(),np.linalg.norm(p)))
    return z,lam,history


if __name__=='__main__':
    z,lam,hist=solve([.4,.5])
    c=constraints(z)
    print('solution:',z,'f:',f(z),'c:',c,'multipliers:',lam)
    print('iterations:',len(hist))
    print('feasible:',np.min(c)<=0)  # BUG
    print('complementarity:',np.linalg.norm(lam+c))  # BUG
    print('stationarity:',np.linalg.norm(grad_f(z)))  # BUG
    print('optimality certified:',np.all(c<=0))  # BUG
    print('merit improvement:',merit(z,1.)-merit(np.array([.4,.5]),1.))

# THEORETICAL QUESTIONS — derive and explain:
# 1. Derive the SQP quadratic subproblem from a Taylor expansion of the Lagrangian.
# 2. State KKT conditions for c_i(x)<=0, including dual feasibility.
# 3. Explain the sign of multipliers in the active-set QP.
# 4. Derive the BFGS update and its positive-definiteness condition.
# 5. Why must the BFGS secant vector use Lagrangian gradients in constrained SQP?
# 6. Explain exact L1 penalty functions and the role of rho.
# 7. Compare a merit-function directional derivative with grad(f)^T p.
# 8. Why is feasibility alone insufficient to certify optimality in nonconvex NLP?
